# Golden Dataset Evaluation
Compares trained model checkpoints on a held-out Golden Dataset - images sourced from google (24 of each produce type, 12 rotten; 12 healthy) independently from the training data to give an honest, out-of-distribution accuracy estimate.

Each model was trained under different dataset / augmentation  conditions; the golden set is fixed across all comparisons so the results are directly comparable.


| Model label | Dataset | Augmentation | Notes |
|---|---|---|---|
| Dirty | Full raw dataset (~29k images) | Yes | Includes near-duplicates and static pre-augmented images |
| No dups+aug | No duplicates | No | --- |
| No dups | No duplicates | No | Baseline clean dataset |

# EXPERIMENTS
### Success criteria
- ID val is over-saturated so selection is based on:
    
    `OOD validation accuracy (primary) => OOD AUC-ROC (Secondary) => OOD ECE (Tertiary).`

### Steps
1. Identify best dataset variation using EfficientNet STL baseline (deduplication, pre-augmentation)

    ***Deduplication was won by ECE tiebreake***
2. Identify best STL architecture at fixed optimizer (AdamW): EfficientNet vs Swin vs MaxViT
3. On winning arch, MTL vs STL with unified stopping criterion (primary-task loss)

4. On winning arch, architectural ablations:
    - freeze / partial-freeze / finetune
    - pretrained / random-init
    - class-weighted / unweighted loss
5. Augmentation ablation on best config from (4)
6. Post-hoc: temperature scaling on val > OOD ECE check

In [ ]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from torchvision.models import get_model, get_weight
from torch.utils.data import DataLoader
from utils.dataset import ProduceDataset
from pathlib import Path
from safetensors.torch import load_file
sys.path.append("..")
from experiment_configs import task_2_config as experiments
from utils.mtl_model import MultiTaskClassifier
from sklearn.metrics import roc_curve, roc_auc_score, brier_score_loss, precision_recall_fscore_support,confusion_matrix
import numpy as np
from plotly.subplots import make_subplots

GOLDEN_PATH =  Path(".") / "data" / "golden_dataset" /"golden_dataset" 
NUM_HEALTH_CLASSES = 2
CLASS_NAMES = ["Healthy", "Rotten"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _load_state_dict(path):
    if path.endswith(".safetensors"):
        return load_file(path, device=str(device))
    ckpt = torch.load(path, weights_only=False, map_location=device)
    return ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt


def load_model(exp, path, num_produce_classes):
    """Rebuild architecture from exp config, then load weights."""
    pretrained_weights = get_weight(exp.weight_string)
    base = get_model(exp.architecture, weights=None)
    if exp.is_mtl:
    # MTL: wrap in MultiTaskClassifier (matches saved state_dict keys)
        model = MultiTaskClassifier(base, num_produce_classes=num_produce_classes,
                                    num_health_classes=NUM_HEALTH_CLASSES)
    else:
        # STL: replace final layer using head_attr pattern
        head_attr = "classifier" if hasattr(base, "classifier") else "head"
        head = getattr(base, head_attr)
        if isinstance(head, nn.Sequential):
            head[-1] = nn.Linear(head[-1].in_features, NUM_HEALTH_CLASSES)
        elif isinstance(head, nn.Linear):
            setattr(base, head_attr, nn.Linear(head.in_features, NUM_HEALTH_CLASSES))
        model = base

    model.load_state_dict(_load_state_dict(path))
    return model.to(device).eval(), pretrained_weights.transforms()


def evaluate(model, transforms, is_mtl):
    dataset = ProduceDataset(dataset_root_dir=GOLDEN_PATH, transform=transforms)
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    all_preds, all_labels, all_probs, all_p_rotten = [], [], [], []
    with torch.no_grad():
        for x, y_health, _ in loader:
            x, y_health = x.to(device), y_health.to(device)
            health_out = model(x)[0] if is_mtl else model(x)
            probs = torch.softmax(health_out, dim=1)
            preds = health_out.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y_health.cpu().tolist())
            all_probs.extend(probs.max(dim=1).values.cpu().tolist())
            all_p_rotten.extend(probs[:, 1].cpu().tolist())   # prob of positive (rotten) class

    return pd.DataFrame({
        "produce":    [Path(p).parent.name.split("__")[0] for p in dataset.image_paths],
        "true_label": all_labels,
        "pred":       all_preds,
        "correct":    [p == l for p, l in zip(all_preds, all_labels)],
        "confidence": all_probs,
        "p_rotten":   all_p_rotten,
    })

def compute_ece(labels, preds, confidences, n_bins=10):
    labels, preds, confidences = map(np.asarray, (labels, preds, confidences))
    edges = np.linspace(0, 1, n_bins + 1)
    ece, bins = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (confidences >= lo) & (confidences < hi) if i < n_bins - 1 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            bins.append({"conf": (lo + hi) / 2, "acc": None, "count": 0})
            continue
        bin_conf = confidences[mask].mean()
        bin_acc = (preds[mask] == labels[mask]).mean()
        bins.append({"conf": bin_conf, "acc": bin_acc, "count": int(mask.sum())})
        ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return ece, bins


In [ ]:
# Define models to evaluate
MODELS = {
#     "EX2 EffNet STL":   (experiments.EX2_EFFICIENTNET_FINETUNE,       "EX2_EFFICIENTNET_FINETUNE_20260420102640.safetensors"),
#     "EX3 EffNet MTL":   (experiments.EX3_EFFICIENTNET_FINETUNE_MTL,   "EX3_EFFICIENTNET_FINETUNE_MTL_20260420124016.safetensors"),
#     "EX5 Swin STL":     (experiments.EX5_SWIN_FINETUNE,               "EX5_SWIN_FINETUNE_20260420152758.safetensors"),
#     "EX6 Swin MTL":     (experiments.EX6_SWIN_FINETUNE_MTL,           "EX6_SWIN_FINETUNE_MTL_20260420164307.safetensors"),
#     "EX8 MaxViT STL":   (experiments.EX8_MAXVIT_FINETUNE,             "EX8_MAXVIT_FINETUNE_20260420181918.safetensors"),
#     "EX9 MaxViT MTL":   (experiments.EX9_MAXVIT_FINETUNE_MTL,         "EX9_MAXVIT_FINETUNE_MTL_20260420203826.safetensors"),
    "EX2_dedup_no_aug": (experiments.EX2_EFFICIENTNET_FINETUNE,         "EX2_dedup_no_aug_EFFICIENTNET_FINETUNE_20260421113156.safetensors"),
    "EX2_dedup_old_aug": (experiments.EX2_EFFICIENTNET_FINETUNE,                "EX2_dedup_EFFICIENTNET_FINETUNE_20260421123836.safetensors"),
    "EX10_dedup_new_aug_7": (experiments.EX10_EFFICIENTNET_FINETUNE_AUG,  "EX10_EFFICIENTNET_FINETUNE_AUG_20260421211057.safetensors"),
    "EX11_dedup_new_aug_10": (experiments.EX11_EFFICIENTNET_FINETUNE_AUG,  "EX11_EFFICIENTNET_FINETUNE_AUG_20260421221546.safetensors"),
    "EX12_dedup_new_aug_13": (experiments.EX12_EFFICIENTNET_FINETUNE_AUG,  "EX12_EFFICIENTNET_FINETUNE_AUG_20260421210924.safetensors"),
}

# Get produce count for MTL
ds = ProduceDataset(dataset_root_dir=GOLDEN_PATH)
num_produce_classes = ds.num_produce_types

# ── Run evaluation ─────────────────────────────────────────────────────────────
results = {}
for name, (exp, path) in MODELS.items():
    print(f"Evaluating: {name}  (arch={exp.architecture}, mtl={exp.is_mtl})")
    model, transforms = load_model(exp, f"models/{path}", num_produce_classes)
    results[name] = evaluate(model, transforms, exp.is_mtl)
    print(f"  Overall: {results[name]['correct'].mean():.4f}")


# ── Compute all metrics first ─────────────────────────────────────────────────
metrics = {}
rows = []
for name, df in results.items():
    fpr, tpr, _ = roc_curve(df["true_label"], df["p_rotten"])
    auc = roc_auc_score(df["true_label"], df["p_rotten"])
    ece, bins = compute_ece(df["true_label"], df["pred"], df["confidence"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        df["true_label"], df["pred"], average="binary", pos_label=1, zero_division=0
    )
    metrics[name] = {"fpr": fpr, "tpr": tpr, "auc": auc, "ece": ece, "bins": bins}
    rows.append({
        "Model":       name,
        "Accuracy":    df["correct"].mean(),
        "AUC-ROC":     auc,
        "ECE":         ece,
        "Brier":       brier_score_loss(df["true_label"], df["p_rotten"]),
        "Mean conf.":  df["confidence"].mean(),
        "Precision":   prec,
        "Recall":      rec,
        "F1":          f1,
        "N":           len(df),
    })

# ── Summary table (before figures) ────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index("Model")
higher_better = ["Accuracy", "AUC-ROC", "F1", "Precision", "Recall"]
lower_better  = ["ECE", "Brier"]

display(summary.style
               .format("{:.4f}", subset=summary.columns.difference(["N"]))
               .highlight_max(subset=higher_better, props="background-color:#2E6F40;")
               .highlight_min(subset=lower_better,  props="background-color:#2E6F40;"))

# ── Overall accuracy bar chart ─────────────────────────────────────────────────
overall = summary[["Accuracy"]].reset_index()
fig1 = px.bar(overall, x="Model", y="Accuracy", text_auto=".3f",
              title="Overall accuracy — golden dataset",
              color="Model", range_y=[0.5, 1.0])
fig1.update_traces(textposition="outside")
fig1.show()

# ── ROC curves ────────────────────────────────────────────────────────────────
fig_roc = go.Figure()
for name, m in metrics.items():
    fig_roc.add_trace(go.Scatter(x=m["fpr"], y=m["tpr"], mode="lines",
                                 name=f"{name} (AUC={m['auc']:.3f})"))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                             line=dict(dash="dash", color="gray"), name="Random"))
fig_roc.update_layout(title="ROC curves — golden dataset",
                      xaxis_title="False positive rate", yaxis_title="True positive rate",
                      height=500, width=700)
fig_roc.show()

# ── Reliability diagram ───────────────────────────────────────────────────────
fig_ece = go.Figure()
for name, m in metrics.items():
    valid = [b for b in m["bins"] if b["acc"] is not None]
    fig_ece.add_trace(go.Scatter(x=[b["conf"] for b in valid],
                                 y=[b["acc"] for b in valid],
                                 mode="lines+markers",
                                 name=f"{name} (ECE={m['ece']:.3f})"))
fig_ece.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                             line=dict(dash="dash", color="gray"), name="Perfect calibration"))
fig_ece.update_layout(title="Reliability diagram — golden dataset",
                      xaxis_title="Confidence", yaxis_title="Accuracy",
                      xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
                      height=500, width=700)
fig_ece.show()
overall = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [df["correct"].mean() for df in results.values()]
})


# ── Per-category grouped bar chart ────────────────────────────────────────────
per_cat = []
for name, df in results.items():
    cat_acc = df.groupby("produce")["correct"].mean().reset_index()
    cat_acc.columns = ["produce", "accuracy"]
    cat_acc["model"] = name
    per_cat.append(cat_acc)

per_cat_df = pd.concat(per_cat)

fig2 = px.bar(per_cat_df, x="produce", y="accuracy", color="model",
              barmode="group", title="Per-category accuracy - golden dataset",
              range_y=[0, 1.0], text_auto=".2f")
fig2.update_layout(xaxis_tickangle=-45, height=500)
fig2.show()

# ── Heatmap ───────────────────────────────────────────────────────────────────
pivot = per_cat_df.pivot(index="model", columns="produce", values="accuracy")

fig3 = px.imshow(pivot, text_auto=".2f", aspect="auto",
                 color_continuous_scale="RdYlGn", range_color=[0.5, 1.0],
                 title="Accuracy heatmap — model vs category")
fig3.show()

# OOD golden-set accuracy - task-critical, this is what the system will actually face
# OOD AUC-ROC - robust to class imbalance, threshold-free. Great tiebreaker when accuracies are within 1-2pp
# OOD ECE/brier - calibration matters because ripeness grading downstream reads softmax confidence
# 
# 1. Identify best dataset variation using EfficientNet STL baseline (deduplication, pre-augmentation)
#    Deduplication (old aug) has wins over no aug; however, deduplicaiton + new aug has superior performance.
#    =DEDUPLICATION (NO AUG) WINS=
# 2. Identify best STL architecture at fixed optimizer (AdamW): EfficientNet vs Swin vs MaxViT
# 3. On winning arch, MTL vs STL with unified stopping criterion (primary-task loss)
# 4. On winning arch, architectural ablations:
#     - freeze / partial-freeze / finetune
#     - pretrained / random-init
#     - class-weighted / unweighted loss
# 5. Augmentation ablation on best config from (4)
# 6. Post-hoc: temperature scaling on val > OOD ECE check


Evaluating: EX2_dedup_no_aug  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8783
Evaluating: EX2_dedup_old_aug  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8826
Evaluating: EX10_dedup_new_aug_7  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8957
Evaluating: EX11_dedup_new_aug_10  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8391
Evaluating: EX12_dedup_new_aug_13  (arch=efficientnet_v2_s, mtl=False)
  Overall: 0.8783


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
EX2_dedup_no_aug,0.8783,0.9641,0.0984,0.1031,0.9766,0.8435,0.9065,0.8739,230
EX2_dedup_old_aug,0.8826,0.9634,0.0429,0.0786,0.9255,0.8636,0.8879,0.8756,230
EX10_dedup_new_aug_7,0.8957,0.9641,0.0582,0.0814,0.9539,0.9029,0.8692,0.8857,230
EX11_dedup_new_aug_10,0.8391,0.9397,0.0908,0.1206,0.9299,0.8723,0.7664,0.8159,230
EX12_dedup_new_aug_13,0.8783,0.9571,0.0729,0.0894,0.9512,0.8496,0.8972,0.8727,230


## 2. Cross-Model Comparison
High-level comparison across all models: overall accuracy, per-category breakdown, and an accuracy heatmap. Key observation: the dirty dataset (largest volume) outperforms cleaner subsets, suggesting data volume dominates over data cleanliness for this task at current scale.

In [ ]:
for model_name, df in results.items():
    y_true = df["true_label"].values
    y_pred = df["pred"].values
    conf   = df["confidence"].values

    # ── Precompute all four panels ──
    cm = confusion_matrix(y_true, y_pred)

    categories = sorted(df["produce"].unique())
    rows = []
    for cat in categories:
        sub = df[df["produce"] == cat]
        p, r, f, _ = precision_recall_fscore_support(
            sub["true_label"], sub["pred"], average="binary", zero_division=0)
        rows.append({"Category": cat, "Precision": p, "Recall": r, "F1": f})
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    rows.append({"Category": "MACRO", "Precision": p_mac, "Recall": r_mac, "F1": f_mac})
    prf_df = pd.DataFrame(rows)

    errors = df[~df["correct"]].copy()
    errors["error_type"] = np.where(errors["true_label"] == 0, "H→R", "R→H")
    total = df.groupby("produce").size()
    breakdown = errors.groupby(["produce", "error_type"]).size().unstack(fill_value=0)
    breakdown_pct = (breakdown.div(total, axis=0) * 100).round(1)

    # ── Build 2x2 grid ──
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Confusion matrix", "Per-category P / R / F1",
                        "Confidence by outcome",  "Error direction per category (%)"),
        horizontal_spacing=0.12, vertical_spacing=0.15,
        column_widths=[0.35, 0.65],
    )

    # 1. Confusion matrix
    fig.add_trace(go.Heatmap(
        z=cm, x=CLASS_NAMES, y=CLASS_NAMES, colorscale="Blues",
        text=cm, texttemplate="%{text}", showscale=False,
    ), row=1, col=1)

    # 2. P/R/F1 per category (grouped bars)
    for metric, color in [("Precision", "#636EFA"), ("Recall", "#EF553B"), ("F1", "#00CC96")]:
        fig.add_trace(go.Bar(
            name=metric, x=prf_df["Category"], y=prf_df[metric],
            marker_color=color, legendgroup=metric,
        ), row=1, col=2)

    # 3. Confidence distribution — violin (plays nice with grouped barmode elsewhere)
    fig.add_trace(go.Violin(
        x=np.where(df["correct"], "Correct", "Incorrect"),
        y=conf, box_visible=True, meanline_visible=True,
        points="outliers", line_color="#333", fillcolor="#AAB7FF",
        showlegend=False,
    ), row=2, col=1)

    # 4. Error direction
    err_long = breakdown_pct.reset_index().melt(id_vars="produce", var_name="error_type", value_name="value")
    for etype, color in [("H→R", "#EF553B"), ("R→H", "#636EFA")]:
        sub = err_long[err_long["error_type"] == etype]
        fig.add_trace(go.Bar(
            name=etype, x=sub["produce"], y=sub["value"],
            marker_color=color, legendgroup=etype,
        ), row=2, col=2)

    fig.update_layout(
        height=750, width=1300,
        title_text=f"<b>{model_name}</b>",
        barmode="group",
    )
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.update_xaxes(tickangle=-45, row=2, col=2)
    fig.update_yaxes(range=[0, 1.05], row=1, col=2)
    fig.update_yaxes(title_text="Confidence", row=2, col=1)

    fig.show()


In [ ]:
from scipy.optimize import minimize_scalar
import torch.nn.functional as F

# ── Temperature scaling calibration ───────────────────────────────────────────
# Calibrate the no-aug model (most overconfident).
# T is found by minimising NLL on the golden set — in practice you'd use a
# separate val set, but this demonstrates the technique on available data.

TARGET_MODEL = "No aug"

def evaluate_with_logits(model, transforms):
    """Same as evaluate() but also returns raw logits for calibration."""
    dataset = ProduceDataset(root_dir=GOLDEN_PATH, transform=transforms)
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            all_logits.append(model(imgs).cpu())
            all_labels.extend(labels.tolist())
    return torch.cat(all_logits), torch.tensor(all_labels)

# Reload the target model and collect logits
m, transforms = load_model(MODELS[TARGET_MODEL])
logits, labels = evaluate_with_logits(m, transforms)

# Find T that minimises NLL on the golden set
def nll(T):
    return F.cross_entropy(logits / T, labels).item()

result   = minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded")
best_T   = result.x
print(f"Optimal temperature T = {best_T:.3f}")

# ── Compare confidence distributions before and after ─────────────────────────
def conf_from_logits(logits, T=1.0):
    probs = torch.softmax(logits / T, dim=1)
    return probs.max(dim=1).values.numpy()

preds_orig = logits.argmax(dim=1).numpy()
correct    = (preds_orig == labels.numpy())

conf_before = conf_from_logits(logits, T=1.0)
conf_after  = conf_from_logits(logits, T=best_T)

fig_cal = make_subplots(rows=1, cols=2,
                        subplot_titles=["Before (T=1)", f"After (T={best_T:.2f})"])

for col, conf, title in [(1, conf_before, "Before"), (2, conf_after, "After")]:
    fig_cal.add_trace(go.Histogram(
        x=conf[correct],  name="Correct",   marker_color="#00CC96",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)
    fig_cal.add_trace(go.Histogram(
        x=conf[~correct], name="Incorrect", marker_color="#EF553B",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)

fig_cal.update_xaxes(range=[0.4, 1.0])
fig_cal.update_layout(barmode="overlay", title=f"{TARGET_MODEL} — confidence before vs after temperature scaling",
                      height=400)
fig_cal.show()

acc = correct.mean()
print(f"Accuracy before: {acc:.4f}")
print(f"Accuracy after:  {acc:.4f}  (unchanged — argmax is T-invariant)")

TypeError: load_model() missing 2 required positional arguments: 'path' and 'num_produce_classes'

## 3. Per-Model Detailed Analysis
For each model: confusion matrix, per-category Precision / Recall / F1, prediction confidence distribution, and error direction breakdown (Healthy→Rotten vs Rotten→Healthy).

The confidence distribution reveals **calibration** — a well-calibrated model should show high confidence on correct predictions and lower confidence on errors. Error direction shows whether the model leans toward false positives (calling rotten produce healthy) or false negatives.